# MyDramaList: Music-Centered K-Drama Discovery
### Kailyn

This notebook collects a targeted sample of music-centered K-dramas. The goal is roughly **100 candidate dramas**.

**Table of Contents**

1. [Setup](#1)
2. [Configure Parameters](#2)
3. [Setting up Caching](#3)
4. [Setting up Parsing](#4)
5. [Extract Tag Ids](#5)
6. [Collect Candidate Dramas](#6)
7. [Add Drama Metadata](#7)
8. [Run Everything](#8)
9. [Summary and Save](#9)

The main idea is that Professor Eni has given us some example keywords to search for, such as singer, idol, band, etc. However, MDL's tag filter takes numeric tag IDs (`th=123`), *not* words, so I (a) search each keyword, (b) open a few top results, and (c) read the IDs of any matching tag links on those pages.

Now, I can run a tag-filtered search per ID and paginate through the results to get the actual list of K-dramas. From there, I'm doing a manual review of the dramas that were flagged (see notebook m3_2).

<a id=1></a>
## 1. Setup

I'm using `seleniumbase` to fetch pages, since MDL uses Cloudflare, so plain `requests` is blocked), and parsing is done with BeautifulSoup on the cached HTML.

In [8]:
import hashlib
import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

import pandas as pd
from bs4 import BeautifulSoup
from seleniumbase import Driver

<a id=2></a>
## 2. Configure Parameters

This notebook can be rerun, should we decide we want to adjust our search.

In [9]:
BASE_URL = "https://mydramalist.com"
SEARCH_URL = f"{BASE_URL}/search"

# Sample size and politeness
TARGET_CANDIDATES = 150      # approximate size of the candidate set
MAX_PAGES_PER_TAG = 10       # safety cap on pagination (20 dramas/page)
REQUEST_DELAY_SECONDS = 1.5  # pause after each *uncached* request (so as not to overwhelm MDL and follow good practices)
ENRICH_WITH_DETAIL_PAGES = True   # also visit each drama's own page to extract metadata (also cached)

# Search filters
DRAMA_TYPE_ID = 68   # On MDL, this is ty=68, to get dramas
COUNTRY_ID = 3       # On MDL, this is co=3, to get South Korea
SORT_BY = "top"      # Alternatively, change to "popular" to get most-watched first

# Discovery keywords
# Used to FIND candidates, not as final inclusion criteria (see top of notebook).
KEYWORDS = [
    "singer", "singing", "idol", "band", "musician", "music industry",
    "guitarist", "choir", "busking", "musical", "music",
    "rapper", "drummer", "pianist", "violinist", "k-pop", "trainee",
]

# \b = word boundary, so "band" matches "Boy Band" but not "Husband".
KEYWORD_PATTERN = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in KEYWORDS) + r")\b", re.IGNORECASE
)
SEEDS_PER_KEYWORD = 3        # top results to open per keyword when harvesting tag IDs

CACHE_DIR = Path("mdl_cache")
CACHE_DIR.mkdir(exist_ok=True)

TAG_IDS_PATH = Path("mdl_tag_ids.json")
REFRESH_TAG_IDS = False

<a id=3></a>
## 3. Cached fetching

This section was written with the help of genAI.

Because I don't know how often we'll need to query MDL, I'm using caching to limit the amount of live work that needs to be done. 

The idea is that every page goes through a big function, `fetch_html(url)`, which checks if the page is already in `mdl_cache/`. If it is, we can read it from the directory, and if it's not, we can load it in the browser, save `page_source`, and return it.

I named filenames with a readable slug (the title) and a short hash of the URL, so that you can easily skim the directory. You can also pass `force=True` or delete the page in the directory to refresh that page.

In [10]:
_driver = None          # one browser for the whole session, created on first use
cache_hits = 0
cache_misses = 0

def get_driver():
    """Create the browser lazily, so a fully cached run never opens one."""
    global _driver
    if _driver is None:
        _driver = Driver(page_load_strategy="eager")
    return _driver

def close_driver():
    global _driver
    if _driver is not None:
        _driver.quit()
        _driver = None

def cache_path(url):
    """Readable-but-unique filename for a URL."""
    parsed = urlparse(url)
    slug = re.sub(r"[^A-Za-z0-9]+", "_", parsed.path + "_" + parsed.query).strip("_")[:60]
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:8]
    return CACHE_DIR / f"{slug}_{digest}.html"

# def looks_like_challenge(html):
#     """True if this is a Cloudflare interstitial rather than a real page."""
#     head = html[:5000].lower()
#     return "just a moment" in head or "cf-challenge" in head or "challenge-platform" in head

def fetch_html(url, force=False):
    """Return the HTML for `url`, from the local cache when possible."""
    global cache_hits, cache_misses
    path = cache_path(url)

    if path.exists() and not force:
        cache_hits += 1
        return path.read_text(encoding="utf-8")

    cache_misses += 1
    driver = get_driver()
    driver.open(url)
    driver.sleep(REQUEST_DELAY_SECONDS)
    html = driver.page_source
    # # Give Cloudflare a few chances to let us through before giving up.
    # for _ in range(3):
    #     if not looks_like_challenge(html):
    #         break
    #     print("  Cloudflare challenge, waiting...")
    #     driver.sleep(5)
    #     html = driver.page_source
    # else:
    #     raise RuntimeError(f"Still blocked by Cloudflare on {url}")
    path.write_text(html, encoding="utf-8")
    return html

<a id=4></a>
## 4. Setting up parsing of search results (aggregation pages)

Most of this code comes from work done in week 2, when we practiced scraping MDL.

In [11]:
PATTERN = re.compile(r"^(.*?)\s*-\s*(\d{4})(?:,\s*(.*))?$")

def parse_search_results(html):
    """Parse one MDL search-results page into a list of drama dicts."""
    soup = BeautifulSoup(html, "html.parser")

    cards = soup.select(".box[id^='mdl-']") or soup.select(".row-cell.content") # I can't remember which one worked

    dramas = []
    for card in cards:
        link = card.select_one(".title a")
        if link is None:
            continue

        title = link.get_text(strip=True)
        url = urljoin(BASE_URL, link.get("href", ""))

        # e.g. "Korean Drama - 2021, 12 episodes"
        drama_type, year, episodes = "", "", ""
        for el in card.select(".text-muted"):
            m = PATTERN.match(el.get_text(" ", strip=True))
            if m:
                drama_type, year, episodes = m.group(1), m.group(2), m.group(3) or ""
                break

        score_el = card.select_one(".score")
        rating = score_el.get_text(strip=True) if score_el else ""

        paragraphs = card.select("p")
        synopsis = paragraphs[-1].get_text(" ", strip=True) if paragraphs else ""

        dramas.append({
            "title": title, "url": url, "type": drama_type, "year": year,
            "episodes": episodes, "rating": rating, "synopsis": synopsis,
        })
    return dramas

<a id=5></a>
## 5. Extract Tag IDs

For each keyword, search MDL, open the top few results, and read any tag links (`...th=<id>`) whose text matches the keyword pattern. This produces a `{tag_id: tag_name}` map, and it also catches related tags like *Singer Female Lead* or *Idol Supporting Character*. All of these pages are cached.

The map is also saved as a JSON, so that you only have to run the conversion once.

In [12]:
def extract_matching_tags(html):
    """Tag links on a drama page whose text matches our keywords -> {id: name}."""
    soup = BeautifulSoup(html, "html.parser")
    found = {}
    for a in soup.select("a[href*='th=']"):
        name = a.get_text(strip=True)
        if not name or not KEYWORD_PATTERN.search(name):
            continue
        for value in parse_qs(urlparse(a["href"]).query).get("th", []):
            if value.isdigit(): # skip comma-lists
                found[value] = name
    return found

def discover_tag_ids():
    tag_ids = {}
    for keyword in KEYWORDS:
        print(f"Keyword search: '{keyword}'")
        results = parse_search_results(fetch_html(f"{SEARCH_URL}?q={keyword}"))
        for drama in results[:SEEDS_PER_KEYWORD]:
            tag_ids.update(extract_matching_tags(fetch_html(drama["url"])))

    print(f"\nDiscovered {len(tag_ids)} music-related tags:")
    for tag_id, name in tag_ids.items():
        print(f"  th={tag_id}: {name}")
    return tag_ids

def save_tag_ids(tag_ids):
    """Write {tag_id: tag_name} plus the keywords that produced it."""
    payload = {"keywords": KEYWORDS, "tags": tag_ids}
    TAG_IDS_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved {len(tag_ids)} tag IDs to {TAG_IDS_PATH}")


def load_or_discover_tag_ids(refresh=REFRESH_TAG_IDS):
    """Use the saved tag IDs if they exist and are still valid; otherwise discover and save."""
    if TAG_IDS_PATH.exists() and not refresh:
        payload = json.loads(TAG_IDS_PATH.read_text(encoding="utf-8"))
        if payload.get("keywords") == KEYWORDS and payload.get("tags"):
            print(f"Loaded {len(payload['tags'])} saved tag IDs from {TAG_IDS_PATH}")
            return payload["tags"]
        print("Saved tag IDs were built from a different KEYWORDS list; re-discovering...")

    tag_ids = discover_tag_ids()
    if tag_ids: # never overwrite the file with an empty result
        save_tag_ids(tag_ids)
    return tag_ids

<a id=6></a>
## 6. Collect Candidate Dramas

Now that we have the tags in id form, we implement pagination, i.e. the `page` query parameter (20 results per page). A tag stops when a page is empty, has nothing new, or hits `MAX_PAGES_PER_TAG`; the whole loop stops at `TARGET_CANDIDATES`.

I implemented pagination as round-robin - scraping page 1 of every tag, then page 2 of every tag, and so on, so that we get a variety of dramas from a variety of tags instead of one huge tag. Any dramas that are repeats are dropped (based on URL), and `matched_tags` records every tag that found each one, since a drama hit by many music tags is probably more central to the music theme.

QUESTION - THE WAY THE NUMBERS WORK, AND SINCE EACH PAGE IS 20 DRAMAS, THE ROUND-ROBIN STYLE STILL DOESN'T GET TO ALL DRAMAS. THOUGHTS?

In [13]:
def build_search_url(tag_id, page):
    url = f"{SEARCH_URL}?adv=titles&th={tag_id}&so={SORT_BY}&page={page}"
    if DRAMA_TYPE_ID is not None:
        url += f"&ty={DRAMA_TYPE_ID}"
    if COUNTRY_ID is not None:
        url += f"&co={COUNTRY_ID}"
    return url


def collect_candidates(tag_ids, target=TARGET_CANDIDATES):
    collected = {}  # url -> record (the dict de-duplicates for us)
    state = {t: {"page": 1, "done": False} for t in tag_ids}

    while len(collected) < target and not all(s["done"] for s in state.values()):
        for tag_id, tag_name in tag_ids.items():
            s = state[tag_id]
            if s["done"] or len(collected) >= target:
                continue

            print(f"[{tag_name}] page {s['page']}  (have {len(collected)}/{target})")
            results = parse_search_results(fetch_html(build_search_url(tag_id, s["page"])))

            new_here = 0
            for drama in results:
                if drama["url"] not in collected:
                    drama["matched_tags"] = [tag_name]
                    collected[drama["url"]] = drama
                    new_here += 1
                elif tag_name not in collected[drama["url"]]["matched_tags"]:
                    collected[drama["url"]]["matched_tags"].append(tag_name)

            s["page"] += 1
            if new_here == 0 or s["page"] > MAX_PAGES_PER_TAG:
                s["done"] = True

    candidates = list(collected.values())[:target]
    print(f"\nCollected {len(candidates)} unique candidates")
    return candidates

<a id=7></a>
## 7. Add Metadata

The search card only shows a little metadata, so I also visited each drama's page to add its details (country, network, aired dates, etc.) and its full tag list.

`music_tags_on_page` is the subset of the drama's tags matching my keywords, and `needs_review` is True when that's empty, so I manually checked those. They all seemed all right.

In [14]:
def parse_details(html):
    """Details block of a drama page -> {label: value}."""
    soup = BeautifulSoup(html, "html.parser")
    details = {}

    # Main-column copy and sidebar copy (same content, different responsive layouts)
    items = (
        soup.select("#show-detailsxx .show-detailsxss .list-item")
        + soup.select(".col-lg-4 .box .box-body .list-item")
    ) or soup.select(".list-item")

    for item in items:
        label_el = item.select_one("b.inline")
        if label_el is None:
            continue
        label = label_el.get_text(strip=True).rstrip(":")
        value = item.get_text(" ", strip=True)[len(label_el.get_text(strip=True)):].strip()
        if label and label not in details:
            details[label] = value
    return details


def parse_tags(html):
    """Full tag list of a drama page (hidden 'show more' tags are in the HTML too)."""
    soup = BeautifulSoup(html, "html.parser")
    links = soup.select(".show-tags a[href*='th=']") or soup.select("a[href*='th=']")
    tags = []
    for a in links:
        name = a.get_text(strip=True)
        if name and name not in tags:
            tags.append(name)
    return tags


def enrich(candidates):
    rows = []
    for i, drama in enumerate(candidates, start=1):
        print(f"[{i}/{len(candidates)}] {drama['title']}")
        record = dict(drama)

        if ENRICH_WITH_DETAIL_PAGES:
            try:
                html = fetch_html(drama["url"])
                record.update(parse_details(html))
                tags = parse_tags(html)
                record["all_tags"] = "|".join(tags)
                record["music_tags_on_page"] = "|".join(t for t in tags if KEYWORD_PATTERN.search(t))
                record["needs_review"] = record["music_tags_on_page"] == ""
            except Exception as e:
                print("  Could not enrich:", e)
                record["needs_review"] = True

        # Flatten the list so it fits in one spreadsheet cell.
        record["matched_tags"] = "|".join(drama["matched_tags"])
        record["n_matched_tags"] = len(drama["matched_tags"])
        rows.append(record)
    return rows

<a id=8></a>
## 8. Run it!

In [15]:
try:
    tag_ids = load_or_discover_tag_ids()
    if not tag_ids:
        raise RuntimeError("No music tags discovered. Inspect a cached drama page in "
                           "mdl_cache/ to check the tag-link selector.")
    candidates = collect_candidates(tag_ids)
    rows = enrich(candidates)
finally:
    close_driver()   # always shut the browser down, even if something failed

print(f"\nCache: {cache_hits} hits, {cache_misses} new requests")
print(f"Pages cached on disk: {len(list(CACHE_DIR.glob('*.html')))}")

Loaded 13 saved tag IDs from mdl_tag_ids.json
[Music Show] page 1  (have 0/150)
[Singing] page 1  (have 0/150)
[Pop Music] page 1  (have 15/150)
[Band] page 1  (have 34/150)
[Singer Supporting Character] page 1  (have 50/150)
[Singer Female Lead] page 1  (have 61/150)
[Choir] page 1  (have 73/150)
[Idol Male Lead] page 1  (have 73/150)
[Hip Hop Music] page 1  (have 87/150)
[Drummer Male Lead] page 1  (have 91/150)
[Musician Male Lead] page 1  (have 93/150)
[Pianist Male Lead] page 1  (have 106/150)
[K-Pop] page 1  (have 118/150)
[Singing] page 2  (have 120/150)
[Pop Music] page 2  (have 120/150)
[Band] page 2  (have 132/150)
[Singer Supporting Character] page 2  (have 146/150)
[Singer Female Lead] page 2  (have 146/150)

Collected 150 unique candidates
[1/150] Jeongnyeon: The Star Is Born
[2/150] Castaway Diva
[3/150] To. Jenny
[4/150] You're Beautiful
[5/150] Namib
[6/150] Sing My Crush
[7/150] Heartstrings
[8/150] The Liar and His Lover
[9/150] Wish You: Your Melody From My Heart
[10

<a id=9></a>
## 9. Summary and Save

The following is simply a DataFrame version of the dramas I scraped, which has been exported to a CSV. `n_matched_tags` is a rough "how music-centered?" signal, and `needs_review` marks rows to check manually, which I did.

In [16]:
df = pd.DataFrame(rows)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.sort_values(["n_matched_tags", "rating"], ascending=False).reset_index(drop=True)

df.to_csv("mdl_music_kdrama_candidates.csv", index=False)

print(f"{len(df)} candidates; {int(df['needs_review'].sum())} flagged for manual review")
df[["title", "year", "rating", "n_matched_tags", "matched_tags", "needs_review"]].head(20)

150 candidates; 38 flagged for manual review


,title,year,rating,n_matched_tags,matched_tags,needs_review
0,Lovely Runner,2024.0,8.9,4,Band|Singer Supporting Character|Idol Male Lea...,True
1,Imitation,2021.0,7.9,4,Pop Music|Singer Female Lead|Idol Male Lead|K-Pop,False
2,Twinkling Watermelon,2023.0,9.2,3,Band|Singer Supporting Character|Musician Male...,False
3,Hospital Playlist Season 2,2021.0,9.0,3,Band|Drummer Male Lead|Musician Male Lead,False
4,Hospital Playlist,2020.0,9.0,3,Band|Drummer Male Lead|Musician Male Lead,False
5,Jeongnyeon: The Star Is Born,2024.0,8.3,3,Singing|Singer Supporting Character|Singer Fem...,False
6,Castaway Diva,2023.0,8.3,3,Singing|Singer Supporting Character|Singer Fem...,False
7,My Lovely Liar,2023.0,8.2,3,Singer Female Lead|Musician Male Lead|Pianist ...,False
8,You're Beautiful,2009.0,8.0,3,Singing|Band|Idol Male Lead,False
9,Jazz for Two,2024.0,7.8,3,Band|Drummer Male Lead|Pianist Male Lead,False


In [17]:
# Quick look at the sample: years and which tags found the most dramas.
print(df["year"].value_counts().sort_index().tail(15), "\n")

tag_counts = (df["matched_tags"].str.split("|").explode().value_counts())
print(tag_counts.head(15))

year
2013.0     3
2014.0     6
2015.0     6
2016.0     7
2017.0     9
2018.0     5
2019.0    10
2020.0    11
2021.0    11
2022.0     4
2023.0    19
2024.0    10
2025.0    12
2026.0     6
2027.0     1
Name: count, dtype: int64 

matched_tags
Band                           36
Pop Music                      32
Singer Female Lead             29
Idol Male Lead                 20
Musician Male Lead             20
K-Pop                          20
Pianist Male Lead              20
Singer Supporting Character    16
Singing                        15
Drummer Male Lead               6
Hip Hop Music                   4
Name: count, dtype: int64
